# Ori – Trợ lý hội thoại bán hàng (RAG + Tool Calling) trên Jupyter / Colab

Notebook này chạy lại toàn bộ project `posora-kb-assistant` (file zip *Mã nguồn và Dữ liệu*) mà không cần Redis Cloud.

| Bước | Nội dung |
|---|---|
| 0 | Cấu hình (chọn backend LLM, khoá API) |
| 1 | Giải nén mã nguồn, cài thư viện, ghi `.env` |
| 2 | Khởi động LiteLLM Gateway (chạy nền) |
| 3 | Thay Redis bằng vector store cục bộ (NumPy) và nạp tri thức |
| 4 | PhoBERT NER: dùng trọng số có sẵn, nếu không có thì huấn luyện lại |
| 5 | Chạy chatbot: demo nhiều lượt và hội thoại tự do |

**Lưu ý:** zip không chứa tệp trọng số PhoBERT (`model.safetensors`), nên Bước 4 sẽ huấn luyện lại một mô hình mới.
Nên chọn runtime **GPU** (Colab: *Runtime → Change runtime type → T4 GPU*).

## Bước 0. Cấu hình

In [ ]:
# ====== CHỈNH Ở ĐÂY ======
BACKEND = "groq"            # "groq" | "gemini" | "ollama"
OLLAMA_MODEL = "qwen2.5:7b" # chỉ dùng khi BACKEND = "ollama" (README: 1.5b quá nhỏ để dựng tham số lồng nhau)
USE_RERANKER = True         # False = bỏ cross-encoder (báo cáo cho thấy tốn ~94 lần độ trễ mà không tăng chất lượng)
RETRAIN_NER = True          # True = huấn luyện lại PhoBERT nếu thiếu trọng số
ZIP_NAME = "Ma__nguo__n_va__Du___lie__u.zip"
# =========================

import os, getpass

def _get_secret(name):
    if os.getenv(name):
        return os.environ[name]
    try:  # Colab Secrets
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return getpass.getpass(f"Nhập {name}: ")

GROQ_API_KEY = GEMINI_API_KEY = "placeholder"
if BACKEND == "groq":
    GROQ_API_KEY = _get_secret("GROQ_API_KEY")
elif BACKEND == "gemini":
    GEMINI_API_KEY = _get_secret("GEMINI_API_KEY")
print("Backend:", BACKEND)

## Bước 1. Giải nén mã nguồn, cài thư viện, ghi `.env`

In [ ]:
import sys, glob, zipfile
from pathlib import Path

WORK = Path("work").resolve()
WORK.mkdir(exist_ok=True)

def _find_zip():
    for pat in (ZIP_NAME, f"/content/{ZIP_NAME}", f"/kaggle/input/**/{ZIP_NAME}", "/mnt/user-data/uploads/" + ZIP_NAME):
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    try:
        from google.colab import files
        print("Chọn file zip để tải lên...")
        up = files.upload()
        return next(iter(up))
    except ImportError:
        raise FileNotFoundError(f"Không thấy {ZIP_NAME}. Đặt file cạnh notebook rồi chạy lại.")

zip_path = _find_zip()
with zipfile.ZipFile(zip_path) as z:
    # bỏ model giọng nói 63 MB (không dùng trong notebook)
    members = [m for m in z.namelist() if not m.endswith(".onnx")]
    z.extractall(WORK, members=members)

PROJ = next(p.parent for p in WORK.rglob("chatbot.py"))
print("PROJ =", PROJ)

In [ ]:
%pip install -q "langchain" langchain-core langchain-community langchain-openai python-dotenv pydantic numpy tiktoken \
    fastembed rank-bm25 unidecode sentence-transformers transformers accelerate "litellm[proxy]" requests

In [ ]:
# Ghi .env (giữ nguyên tên biến của project; Redis không dùng)
ALIAS = {
    "groq":   ("dmp-chat",        "dmp-intent"),
    "gemini": ("dmp-chat-gemini", "dmp-intent-gemini"),
    "ollama": ("dmp-chat-local",  "dmp-intent-local"),
}[BACKEND]

env = {
    "GROQ_API_KEY": GROQ_API_KEY,
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "LITELLM_MASTER_KEY": "sk-local-notebook",
    "LLM_GATEWAY_URL": "http://localhost:4000/v1",
    "LLM_GATEWAY_MODEL": ALIAS[0],
    "LLM_GATEWAY_INTENT_MODEL": ALIAS[1],
    "LLM_GATEWAY_TIMEOUT_SECONDS": "120",
    "LLM_CHAT_TEMPERATURE": "0.7",
    "LLM_CHAT_MAX_TOKENS": "300",
    "LLM_INTENT_TEMPERATURE": "0",
    "LLM_INTENT_MAX_TOKENS": "200",
    "GATEWAY_GROQ_MODEL": "groq/openai/gpt-oss-20b",
    "GATEWAY_GEMINI_MODEL": "gemini/gemini-2.5-flash",
    "GATEWAY_LOCAL_MODEL": f"ollama/{OLLAMA_MODEL}",
    "GATEWAY_LOCAL_API_BASE": "http://localhost:11434",
    "CONTEXT_STORE_BACKEND": "memory",
    "CONTEXT_MAX_INPUT_TOKENS": "8192",
    "CONTEXT_RESERVED_OUTPUT_TOKENS": "768",
    "CONTEXT_MAX_HISTORY_TOKENS": "3200",
    "CONTEXT_MAX_EXTERNAL_TOKENS": "3200",
    "PYTHONUTF8": "1",
}
(PROJ / ".env").write_text("\n".join(f"{k}={v}" for k, v in env.items()) + "\n", encoding="utf-8")
os.environ.update(env)
print(".env đã ghi tại", PROJ / ".env")

## Bước 2. Khởi động LiteLLM Gateway (chạy nền)

In [ ]:
import subprocess, shutil, time, requests

# (Chỉ backend "ollama") cài và chạy Ollama cục bộ
if BACKEND == "ollama":
    if not shutil.which("ollama"):
        !apt-get -qq install -y zstd > /dev/null
        !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"], stdout=open(WORK / "ollama.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(5)
    !ollama pull {OLLAMA_MODEL}

def start_gateway(port=4000, timeout=180):
    exe = shutil.which("litellm")
    cmd = [exe] if exe else [sys.executable, "-c", "from litellm.proxy.proxy_cli import run_server; run_server()"]
    cmd += ["--config", str(PROJ / "llm_gateway" / "config.yaml"), "--port", str(port)]
    log = open(WORK / "gateway.log", "w")
    proc = subprocess.Popen(cmd, cwd=PROJ, env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT)
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            break
        try:
            if requests.get(f"http://localhost:{port}/health/liveliness", timeout=2).ok:
                return proc
        except requests.RequestException:
            pass
        time.sleep(2)
    print((WORK / "gateway.log").read_text()[-3000:])
    raise RuntimeError("Gateway không khởi động được, xem log ở trên.")

gateway_proc = start_gateway()
print("Gateway đã sẵn sàng.")

In [ ]:
# Kiểm tra nhanh gateway -> nhà cung cấp LLM
r = requests.post("http://localhost:4000/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.environ['LITELLM_MASTER_KEY']}"},
    json={"model": os.environ["LLM_GATEWAY_MODEL"], "max_tokens": 200,
          "messages": [{"role": "user", "content": "Chào bạn, trả lời đúng một câu ngắn."}]}, timeout=120)
print(r.status_code, r.text[:400])

## Bước 3. Vector store cục bộ thay Redis
Project gốc dùng `langchain-redis` (Redis Cloud). Cell dưới **không sửa file nguồn**: nó đăng ký một module `rag.vector_store` thay thế
(NumPy, độ đo cosine, điểm trả về là cosine similarity giống bản gốc), trước khi `rag` được import.

In [ ]:
import types, hashlib
import numpy as np
from langchain_core.documents import Document
from langchain_core.vectorstores import VectorStore

os.chdir(PROJ)
if str(PROJ) not in sys.path:
    sys.path.insert(0, str(PROJ))

class LocalVectorStore(VectorStore):
    """Vector store trong bộ nhớ, cosine similarity (càng cao càng giống)."""
    def __init__(self, embedding):
        self._emb = embedding
        self._docs: list[Document] = []
        self._mat = None

    @property
    def embeddings(self):
        return self._emb

    @classmethod
    def from_texts(cls, texts, embedding, metadatas=None, **kwargs):
        s = cls(embedding)
        s.add_texts(texts, metadatas)
        return s

    def add_texts(self, texts, metadatas=None, **kwargs):
        metadatas = metadatas or [{} for _ in texts]
        return self.add_documents([Document(page_content=t, metadata=m) for t, m in zip(texts, metadatas)])

    def add_documents(self, documents, **kwargs):  # tham số ids được bỏ qua
        vecs = np.asarray(self._emb.embed_documents([d.page_content for d in documents]), dtype="float32")
        vecs /= (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12)
        self._docs += list(documents)
        self._mat = vecs if self._mat is None else np.vstack([self._mat, vecs])
        return [str(i) for i in range(len(self._docs))]

    def similarity_search_with_score(self, query, k=4, **kwargs):
        if self._mat is None:
            return []
        q = np.asarray(self._emb.embed_query(query), dtype="float32")
        q /= (np.linalg.norm(q) + 1e-12)
        sims = self._mat @ q
        top = np.argsort(-sims)[:k]
        return [(self._docs[i], float(sims[i])) for i in top]

    def similarity_search(self, query, k=4, **kwargs):
        return [d for d, _ in self.similarity_search_with_score(query, k)]

_store = {"obj": None}

def _get_vector_store():
    if _store["obj"] is None:
        from rag.embeddings import get_embeddings
        _store["obj"] = LocalVectorStore(get_embeddings())
    return _store["obj"]

def _reset_collection():
    _store["obj"] = None

_mod = types.ModuleType("rag.vector_store")
_mod.get_vector_store = _get_vector_store
_mod.reset_collection = _reset_collection
_mod.LocalVectorStore = LocalVectorStore
sys.modules["rag.vector_store"] = _mod   # phải chạy trước khi import rag
print("Đã đăng ký rag.vector_store (cục bộ).")

In [ ]:
import rag.hybrid_retriever as _hr

if not USE_RERANKER:
    class _PassThroughReranker:
        def rerank_with_scores(self, query, documents, top_k):
            return [(d, 0.0) for d in list(documents)[:top_k]]
    _hr.get_reranker = lambda: _PassThroughReranker()
    print("Reranker: TẮT (giữ thứ tự RRF)")
else:
    print("Reranker: BẬT (BAAI/bge-reranker-v2-m3, tải ~2 GB ở lần gọi đầu)")

from rag.ingest import build_index
build_index()   # lần đầu tải multilingual-e5-large (~2,2 GB) nên mất vài phút

In [ ]:
from rag import retrieve
# Câu cuối là câu NGOÀI phạm vi: theo báo cáo (mục 5.4.4) với ngưỡng 0,40/0,55 hệ thống vẫn trả về ngữ cảnh (dương tính giả 100%).
for q in ["Quán có món cà phê muối không, giá bao nhiêu?", "Quán mở cửa lúc mấy giờ?", "Quán có bán xe máy cũ không?"]:
    out = retrieve(q)
    print("Q:", q)
    print(out["context"] or "(không có ngữ cảnh)", "\n")

## Bước 4. PhoBERT NER
`ner_model/best_phobert_ner/` trong zip chỉ có cấu hình và tokenizer, **không có trọng số**. Nếu thiếu, cell dưới sinh một tập gán nhãn BIO từ chính `menu.json` / `menu.md` mục VII / `faq.md`
rồi tinh chỉnh `vinai/phobert-base` (4 epoch, lr 3e-5, batch 16, seed 42 như báo cáo).

> Đây là bản cài đặt lại gọn của bộ sinh dữ liệu: **không phải** script `eval/gen_ner_dataset.py` gốc (không có trong zip), nên số đo có thể khác báo cáo.
> Số F1 trên tập kiểm thử sinh cùng nguồn sẽ rất cao vì cùng mẫu câu; không dùng nó để kết luận khả năng khái quát.

In [ ]:
import json, random, re

NER_DIR = PROJ / "ner_model" / "best_phobert_ner"
RETRAINED_DIR = PROJ / "ner_model" / "phobert_ner_retrained"
has_weights = any((NER_DIR / n).exists() for n in ("model.safetensors", "pytorch_model.bin"))
print("Có trọng số gốc:", has_weights)

LABELS = ["O"] + [f"{p}-{t}" for t in ["PRODUCT","SIZE","QUANTITY","ICE_LEVEL","SUGAR_LEVEL","TOPPING","MILK_ALTERNATIVE"] for p in ("B","I")]
L2I = {l: i for i, l in enumerate(LABELS)}   # cùng thứ tự với labels.json của mô hình gốc

def gen_dataset(n=3600, seed=42):
    rnd = random.Random(seed)
    menu = json.load(open(PROJ / "data" / "menu.json", encoding="utf-8"))
    names = [m["name"] for m in menu]
    pre = ["cho tôi", "cho mình", "cho em", "lấy giúp em", "thêm", "đặt", "mình muốn đặt", "order", "em lấy", "gọi giúp mình", "cho anh"]
    post = ["nha", "nhé", "giúp mình", "ạ", ""]
    num_w = ["một", "hai", "ba", "bốn", "năm"]
    sizes = [("size", "L"), ("size", "M"), ("size", "lớn"), ("size", "nhỏ"), ("cỡ", "lớn"), ("ly", "to"), ("ly", "lớn"), ("", "L"), ("", "M")]
    ices = ["ít đá", "không đá", "50% đá", "70% đá", "30% đá", "đá riêng"]
    sugars = ["ít đường", "không đường", "50% đường", "70% đường", "30% đường"]
    toppings = ["trân châu đen", "trân châu hoàng kim", "thạch cà phê", "thạch phô mai", "kem muối", "kem cheese", "màng macchiato", "extra espresso shot"]
    milks = ["sữa hạnh nhân", "sữa đậu nành", "sữa hạt"]

    def words(text, label=None):
        ws = text.split()
        if label is None:
            return [(w, "O") for w in ws]
        return [(w, ("B-" if i == 0 else "I-") + label) for i, w in enumerate(ws)]

    def item():
        seg = []
        q = str(rnd.randint(1, 5)) if rnd.random() < .5 else rnd.choice(num_w)
        seg += words(q, "QUANTITY")
        if rnd.random() < .5:
            seg += words("ly")
        name = rnd.choice(names)
        seg += words(name.lower() if rnd.random() < .6 else name, "PRODUCT")
        if rnd.random() < .75:
            sw, sv = rnd.choice(sizes)
            if sw:
                seg += words(sw)
            seg += words(sv, "SIZE")
        if rnd.random() < .25:
            seg += words(rnd.choice(ices), "ICE_LEVEL")
        if rnd.random() < .25:
            seg += words(rnd.choice(sugars), "SUGAR_LEVEL")
        if rnd.random() < .2:
            seg += words(rnd.choice(["thêm", "kèm"])) + words(rnd.choice(toppings), "TOPPING")
        if rnd.random() < .12:
            seg += words("đổi sang") + words(rnd.choice(milks), "MILK_ALTERNATIVE")
        return seg

    out, seen = [], set()
    while len(out) < n:
        r = rnd.random()
        if r < .72:
            s = words(rnd.choice(pre))
            s += item()
            if rnd.random() < .25:
                s += words("và") + item()
            s += words(rnd.choice(post))
        elif r < .82:
            s = words(rnd.choice(["xoá", "bỏ", "huỷ", "không lấy nữa"]))
            nm = rnd.choice(names)
            s += words(nm.lower(), "PRODUCT")
            if rnd.random() < .6:
                sw, sv = rnd.choice(sizes[:2])
                s += words("size") + words(sv, "SIZE")
        elif r < .9:
            s = words(rnd.choice(["xem giỏ hàng", "giỏ hàng của mình có gì", "tổng tiền là bao nhiêu"]))
        elif r < .95:
            s = words(rnd.choice(["thanh toán", "chốt đơn giúp mình", "tính tiền nhé"]))
        else:
            s = words(rnd.choice(["quán mở cửa lúc mấy giờ", "có chỗ đậu ô tô không", "quán có wifi không", "có giao hàng không"]))
        key = " ".join(w for w, _ in s).strip()
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(s)
    return out

def spans(tags):
    res, cur = set(), None
    for i, t in enumerate(list(tags) + ["O"]):
        if t.startswith("B-") or t == "O" or (t.startswith("I-") and (cur is None or cur[0] != t[2:])):
            if cur:
                res.add((cur[0], cur[1], i))
            cur = (t[2:], i) if t != "O" else None
    return res

In [ ]:
def train_ner():
    import torch, inspect, numpy as np
    from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer

    data = gen_dataset()
    random.Random(42).shuffle(data)
    n1, n2 = int(.7 * len(data)), int(.85 * len(data))
    splits = {"train": data[:n1], "dev": data[n1:n2], "test": data[n2:]}
    print({k: len(v) for k, v in splits.items()})

    tok = AutoTokenizer.from_pretrained("vinai/phobert-base")

    def encode(sent, max_len=128):
        ids, lab = [tok.cls_token_id], [-100]
        for w, l in sent:
            pieces = tok.tokenize(w) or [tok.unk_token]
            pid = tok.convert_tokens_to_ids(pieces)
            cont = ("I-" + l[2:]) if l != "O" else "O"       # mọi subword đều có nhãn (tương thích pipeline gộp thực thể)
            ids += pid
            lab += [L2I[l]] + [L2I[cont]] * (len(pid) - 1)
        ids, lab = ids[: max_len - 1] + [tok.sep_token_id], lab[: max_len - 1] + [-100]
        return {"input_ids": ids, "labels": lab}

    enc = {k: [encode(s) for s in v] for k, v in splits.items()}

    class DS(torch.utils.data.Dataset):
        def __init__(self, rows): self.rows = rows
        def __len__(self): return len(self.rows)
        def __getitem__(self, i): return self.rows[i]

    def collate(batch):
        m = max(len(b["input_ids"]) for b in batch)
        ids = torch.full((len(batch), m), tok.pad_token_id); lab = torch.full((len(batch), m), -100); att = torch.zeros((len(batch), m), dtype=torch.long)
        for i, b in enumerate(batch):
            n = len(b["input_ids"])
            ids[i, :n] = torch.tensor(b["input_ids"]); lab[i, :n] = torch.tensor(b["labels"]); att[i, :n] = 1
        return {"input_ids": ids, "attention_mask": att, "labels": lab}

    def compute_metrics(p):
        pred, gold = p
        pred = pred.argmax(-1)
        tp = fp = fn = 0
        for pr, gl in zip(pred, gold):
            keep = gl != -100
            ps, gs = spans([LABELS[i] for i in pr[keep]]), spans([LABELS[i] for i in gl[keep]])
            tp += len(ps & gs); fp += len(ps - gs); fn += len(gs - ps)
        P = tp / (tp + fp + 1e-9); R = tp / (tp + fn + 1e-9)
        return {"precision": P, "recall": R, "f1": 2 * P * R / (P + R + 1e-9)}

    model = AutoModelForTokenClassification.from_pretrained(
        "vinai/phobert-base", num_labels=len(LABELS),
        id2label=dict(enumerate(LABELS)), label2id=L2I)

    kw = dict(output_dir=str(WORK / "ner_run"), num_train_epochs=4, per_device_train_batch_size=16, per_device_eval_batch_size=32,
              learning_rate=3e-5, weight_decay=0.01, lr_scheduler_type="linear", warmup_steps=0, seed=42,
              save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1", save_total_limit=1,
              report_to="none", fp16=torch.cuda.is_available(), logging_steps=50)
    key = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"
    kw[key] = "epoch"
    trainer = Trainer(model=model, args=TrainingArguments(**kw), train_dataset=DS(enc["train"]), eval_dataset=DS(enc["dev"]),
                      data_collator=collate, compute_metrics=compute_metrics)
    trainer.train()
    print("TEST:", trainer.evaluate(DS(enc["test"]), metric_key_prefix="test"))

    RETRAINED_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(RETRAINED_DIR))
    tok.save_pretrained(str(RETRAINED_DIR))
    (RETRAINED_DIR / "labels.json").write_text(json.dumps({"id2label": {str(i): l for i, l in enumerate(LABELS)}}, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Đã lưu mô hình tại", RETRAINED_DIR)

if not has_weights and RETRAIN_NER:
    train_ner()
if RETRAINED_DIR.exists():
    os.environ["PHOBERT_NER_MODEL_DIR"] = str(RETRAINED_DIR)

In [ ]:
# Kiểm tra runtime NER của project. Nếu pipeline chuẩn của transformers không chạy được với tokenizer PhoBERT
# (cần fast tokenizer để gộp thực thể), chuyển sang bộ dự đoán thủ công theo từ.
import intent.phobert_runtime as pr
TEST_SENT = "cho mình 2 ly cà phê đen đá size lớn ít đường nha"

def _manual_runtime_patch():
    import torch
    from transformers import AutoModelForTokenClassification, AutoTokenizer

    class ManualNERPipeline:
        def __init__(self, model, tok):
            self.model, self.tok = model.eval(), tok
            self.dev = next(model.parameters()).device
        def __call__(self, text):
            words = text.split()
            ids, first = [self.tok.cls_token_id], []
            for w in words:
                pid = self.tok.convert_tokens_to_ids(self.tok.tokenize(w) or [self.tok.unk_token])
                first.append(len(ids)); ids += pid
            ids.append(self.tok.sep_token_id)
            with torch.no_grad():
                logits = self.model(input_ids=torch.tensor([ids], device=self.dev)).logits[0]
            tags = [self.model.config.id2label[int(logits[i].argmax())] for i in first]
            out, cur = [], None
            for w, t in zip(words, tags):
                if t == "O":
                    cur = None; continue
                typ = t[2:]
                if t.startswith("B-") or cur is None or cur["entity_group"] != typ:
                    cur = {"entity_group": typ, "word": w}; out.append(cur)
                else:
                    cur["word"] += " " + w
            return out

    def _build(model_dir):
        tok = AutoTokenizer.from_pretrained(str(model_dir))
        model = AutoModelForTokenClassification.from_pretrained(str(model_dir))
        pr._configure_labels(model, model_dir)
        model.to("cuda" if torch.cuda.is_available() else "cpu")
        return pr.PhoBERTRuntime(model_dir, tok, model, ManualNERPipeline(model, tok))
    pr._get_runtime_for_dir.cache_clear()
    pr._build_runtime = _build

try:
    print(pr.get_phobert_runtime().extract(TEST_SENT))
except FileNotFoundError as e:
    print("Chưa có mô hình NER ->", e)
except Exception as e:
    print("Pipeline chuẩn lỗi:", repr(e)[:200], "-> dùng bộ dự đoán thủ công")
    _manual_runtime_patch()
    print(pr.get_phobert_runtime().extract(TEST_SENT))

## Bước 5. Chạy chatbot

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)
from chatbot import ask, reset_history

SID = "demo"
reset_history(SID)
demo = [
    "Quán có món cà phê muối không, giá bao nhiêu?",
    "Cho mình 2 ly Cà Phê Đen Đá size L",
    "Thêm 1 ly Trà Sữa Ô Long Nướng",      # thiếu size -> Ori hỏi lại
    "size M nha",                           # bổ sung khe ở lượt sau
    "Xem giỏ hàng giúp mình",
    "Thanh toán",
]
for q in demo:
    print("👤", q)
    print("🤖 Ori:", ask(q, session_id=SID), "\n")

In [ ]:
# Hội thoại tự do. Gõ "exit" để thoát, "reset" để xoá lịch sử và giỏ hàng.
SID = "free"
while True:
    q = input("👤 Bạn: ").strip()
    if not q:
        continue
    if q.lower() == "exit":
        break
    if q.lower() == "reset":
        reset_history(SID); print("(đã reset)"); continue
    print("🤖 Ori:", ask(q, session_id=SID), "\n")

## Bước 5c. Giao diện Streamlit
Streamlit chạy ở **tiến trình riêng**, nên tự dựng lại vector store, reranker và bản vá NER (file `st_app/ori_bootstrap.py`). Nó dùng lại **Gateway** (Bước 2) và **mô hình NER đã huấn luyện** (Bước 4) đang có trên đĩa.

**Thứ tự chạy:** Cell 2 → 4 (cấu hình, giải nén) → cài thư viện → ghi `.env` → Bước 2 (Gateway) → Bước 4 (cell sinh dữ liệu + cell `train_ner`) → **hai cell dưới đây**.

**BỎ QUA** các cell Bước 3, kiểm tra `retrieve`, kiểm tra NER thủ công, demo và hội thoại `input()` — Streamlit tự làm, và chạy cả hai sẽ nạp mô hình hai lần, dễ hết RAM Colab.

Phải chạy cell `PROJ`/`WORK` (Bước 1) trong cùng phiên này vì cell dưới dùng biến `WORK`, `PROJ`, `USE_RERANKER`.

In [ ]:
BOOTSTRAP = r'''
import os, sys, types
from pathlib import Path
import numpy as np
from langchain_core.documents import Document
from langchain_core.vectorstores import VectorStore


class LocalVectorStore(VectorStore):
    """Vector store trong bo nho, cosine similarity (cang cao cang giong)."""
    def __init__(self, embedding):
        self._emb = embedding
        self._docs = []
        self._mat = None

    @property
    def embeddings(self):
        return self._emb

    @classmethod
    def from_texts(cls, texts, embedding, metadatas=None, **kwargs):
        s = cls(embedding)
        s.add_texts(texts, metadatas)
        return s

    def add_texts(self, texts, metadatas=None, **kwargs):
        metadatas = metadatas or [{} for _ in texts]
        return self.add_documents([Document(page_content=t, metadata=m) for t, m in zip(texts, metadatas)])

    def add_documents(self, documents, **kwargs):
        vecs = np.asarray(self._emb.embed_documents([d.page_content for d in documents]), dtype="float32")
        vecs /= (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12)
        self._docs += list(documents)
        self._mat = vecs if self._mat is None else np.vstack([self._mat, vecs])
        return [str(i) for i in range(len(self._docs))]

    def similarity_search_with_score(self, query, k=4, **kwargs):
        if self._mat is None:
            return []
        q = np.asarray(self._emb.embed_query(query), dtype="float32")
        q /= (np.linalg.norm(q) + 1e-12)
        sims = self._mat @ q
        top = np.argsort(-sims)[:k]
        return [(self._docs[i], float(sims[i])) for i in top]

    def similarity_search(self, query, k=4, **kwargs):
        return [d for d, _ in self.similarity_search_with_score(query, k)]


def _dereference_fastembed_cache():
    """onnxruntime moi tu choi 'External data path escapes model directory' khi model.onnx va
    model.onnx_data la symlink tro vao cac thu muc blobs khac nhau. Thay moi symlink trong
    snapshots/ bang file that (di chuyen blob, khong ton them dung luong)."""
    import shutil, tempfile
    cache = os.environ.get("FASTEMBED_CACHE_PATH") or os.path.join(tempfile.gettempdir(), "fastembed_cache")
    fixed = 0
    for dirpath, dirnames, filenames in os.walk(cache):
        if "snapshots" not in Path(dirpath).parts:
            continue
        for name in filenames:
            link = os.path.join(dirpath, name)
            if not os.path.islink(link):
                continue
            target = os.path.realpath(link)
            if not os.path.isfile(target):
                continue
            os.unlink(link)
            shutil.move(target, link)
            fixed += 1
    return fixed


def _load_embeddings():
    from rag.embeddings import get_embeddings
    try:
        return get_embeddings()
    except Exception as e:
        msg = str(e)
        if "External data path" not in msg and "escapes model directory" not in msg:
            raise
        _dereference_fastembed_cache()
        return get_embeddings()


def _register_vector_store():
    store = {"obj": None}

    def get_vector_store():
        if store["obj"] is None:
            store["obj"] = LocalVectorStore(_load_embeddings())
        return store["obj"]

    def reset_collection():
        store["obj"] = None

    mod = types.ModuleType("rag.vector_store")
    mod.get_vector_store = get_vector_store
    mod.reset_collection = reset_collection
    mod.LocalVectorStore = LocalVectorStore
    sys.modules["rag.vector_store"] = mod   # phai chay truoc khi import rag


_DEFAULTS = {
    "LLM_GATEWAY_TIMEOUT_SECONDS": "120",
    "LLM_CHAT_TEMPERATURE": "0.7",
    "LLM_CHAT_MAX_TOKENS": "300",
    "LLM_INTENT_TEMPERATURE": "0",
    "LLM_INTENT_MAX_TOKENS": "200",
    "CONTEXT_STORE_BACKEND": "memory",
    "CONTEXT_MAX_INPUT_TOKENS": "8192",
    "CONTEXT_RESERVED_OUTPUT_TOKENS": "768",
    "CONTEXT_MAX_HISTORY_TOKENS": "3200",
    "CONTEXT_MAX_EXTERNAL_TOKENS": "3200",
    "PYTHONUTF8": "1",
}


def configure_env(secrets=None):
    """Chuan bi bien moi truong khi chay ngoai notebook (vd Streamlit Cloud: khong co Gateway / .env).
    Tra ve chuoi loi (tieng Viet) neu thieu khoa API, nguoc lai tra ve None."""
    for k, v in (secrets or {}).items():
        if isinstance(v, (str, int, float, bool)):
            os.environ.setdefault(str(k), str(v))
    for k, v in _DEFAULTS.items():
        os.environ.setdefault(k, v)
    if not os.environ.get("LLM_GATEWAY_URL"):
        # Khong co LiteLLM Gateway -> goi thang Groq (API tuong thich OpenAI)
        key = os.environ.get("GROQ_API_KEY", "").strip()
        if not key or key == "placeholder":
            return ("Thiếu GROQ_API_KEY. Vào Manage app → Settings → Secrets và thêm dòng: "
                    'GROQ_API_KEY = "gsk_..."')
        model = os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile")
        os.environ["LLM_GATEWAY_URL"] = "https://api.groq.com/openai/v1"
        os.environ["LITELLM_MASTER_KEY"] = key
        os.environ["LLM_GATEWAY_MODEL"] = model
        os.environ["LLM_GATEWAY_INTENT_MODEL"] = model
    return None


def _patch_ner():
    try:
        return _patch_ner_impl()
    except ImportError as e:
        return "NER: tat (thieu thu vien: " + str(e)[:60] + ")"
    except Exception as e:
        return "NER: tat (" + repr(e)[:80] + ")"


def _patch_ner_impl():
    import intent.phobert_runtime as pr
    test = "cho minh 2 ly ca phe den da size lon it duong nha"
    try:
        pr.get_phobert_runtime().extract(test)
        return "NER: pipeline chuan OK"
    except FileNotFoundError as e:
        return "NER: chua co mo hinh (" + str(e)[:80] + ")"
    except Exception:
        pass

    import torch
    from transformers import AutoModelForTokenClassification, AutoTokenizer

    class ManualNERPipeline:
        def __init__(self, model, tok):
            self.model, self.tok = model.eval(), tok
            self.dev = next(model.parameters()).device
        def __call__(self, text):
            words = text.split()
            ids, first = [self.tok.cls_token_id], []
            for w in words:
                pid = self.tok.convert_tokens_to_ids(self.tok.tokenize(w) or [self.tok.unk_token])
                first.append(len(ids)); ids += pid
            ids.append(self.tok.sep_token_id)
            with torch.no_grad():
                logits = self.model(input_ids=torch.tensor([ids], device=self.dev)).logits[0]
            tags = [self.model.config.id2label[int(logits[i].argmax())] for i in first]
            out, cur = [], None
            for w, t in zip(words, tags):
                if t == "O":
                    cur = None; continue
                typ = t[2:]
                if t.startswith("B-") or cur is None or cur["entity_group"] != typ:
                    cur = {"entity_group": typ, "word": w}; out.append(cur)
                else:
                    cur["word"] += " " + w
            return out

    def _build(model_dir):
        tok = AutoTokenizer.from_pretrained(str(model_dir))
        model = AutoModelForTokenClassification.from_pretrained(str(model_dir))
        pr._configure_labels(model, model_dir)
        model.to("cuda" if torch.cuda.is_available() else "cpu")
        return pr.PhoBERTRuntime(model_dir, tok, model, ManualNERPipeline(model, tok))

    pr._get_runtime_for_dir.cache_clear()
    pr._build_runtime = _build
    try:
        pr.get_phobert_runtime().extract(test)
        return "NER: dung bo du doan thu cong"
    except Exception as e:
        return "NER: loi " + repr(e)[:120]


def setup(proj, use_reranker=True):
    proj = str(proj)
    os.chdir(proj)
    if proj not in sys.path:
        sys.path.insert(0, proj)
    try:
        from dotenv import load_dotenv
        load_dotenv(Path(proj) / ".env")
    except Exception:
        pass

    _register_vector_store()

    import rag.hybrid_retriever as hr
    if not use_reranker:
        class _PassThroughReranker:
            def rerank_with_scores(self, query, documents, top_k):
                return [(d, 0.0) for d in list(documents)[:top_k]]
        hr.get_reranker = lambda: _PassThroughReranker()

    from rag.ingest import build_index
    build_index()

    ner_status = _patch_ner()

    import logging
    logging.basicConfig(level=logging.WARNING)
    from chatbot import ask, reset_history
    return ask, reset_history, ner_status
'''

APP = r'''
import os, sys, uuid
from pathlib import Path
import streamlit as st

sys.path.insert(0, str(Path(__file__).parent))
import ori_bootstrap

st.set_page_config(page_title="Ori - Trợ lý bán hàng", page_icon="☕")

try:
    _secrets = dict(st.secrets)
except Exception:
    _secrets = {}

_err = ori_bootstrap.configure_env(_secrets)
if _err:
    st.error(_err)
    st.stop()

PROJ = os.environ.get("ORI_PROJ") or str(Path(__file__).resolve().parent.parent / "posora-kb-assistant")
USE_RERANKER = os.environ.get("ORI_USE_RERANKER", "0") == "1"


@st.cache_resource(show_spinner="Đang nạp mô hình và tri thức (lần đầu mất vài phút)...")
def load():
    return ori_bootstrap.setup(PROJ, USE_RERANKER)


ask, reset_history, ner_status = load()

if "sid" not in st.session_state:
    st.session_state.sid = "st-" + uuid.uuid4().hex[:8]
    st.session_state.msgs = []
    reset_history(st.session_state.sid)


def _reset():
    reset_history(st.session_state.sid)
    st.session_state.msgs = []


EXAMPLES = [
    "Quán có món cà phê muối không, giá bao nhiêu?",
    "Cho mình 2 ly Cà Phê Đen Đá size L",
    "Xem giỏ hàng giúp mình",
    "Thanh toán",
]

with st.sidebar:
    st.header("Ori")
    st.button("🗑️ Xoá hội thoại & giỏ hàng", on_click=_reset, use_container_width=True)
    st.caption("Câu hỏi mẫu")
    for ex in EXAMPLES:
        if st.button(ex, use_container_width=True):
            st.session_state.pending = ex
    st.caption(ner_status)

st.title("☕ Ori – Trợ lý hội thoại bán hàng")

for m in st.session_state.msgs:
    with st.chat_message(m["role"]):
        st.markdown(m["content"])

prompt = st.chat_input("Nhập tin nhắn...") or st.session_state.pop("pending", None)
if prompt:
    st.session_state.msgs.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)
    with st.chat_message("assistant"):
        with st.spinner("Ori đang trả lời..."):
            try:
                reply = ask(prompt, session_id=st.session_state.sid)
            except Exception as e:
                reply = f"⚠️ Lỗi: {e}"
        st.markdown(reply)
    st.session_state.msgs.append({"role": "assistant", "content": reply})
'''


In [ ]:
# Ghi 2 file của ứng dụng Streamlit, chạy Streamlit nền, mở link công khai bằng cloudflared
import subprocess, re, time, os, sys, stat, urllib.request, requests

# Gateway (Bước 2) phải đang chạy
try:
    assert requests.get("http://localhost:4000/health/liveliness", timeout=3).ok
except Exception:
    raise RuntimeError("Gateway chưa chạy. Hãy chạy Bước 2 (Cell start_gateway) trước.")

APP_DIR = WORK / "st_app"
APP_DIR.mkdir(exist_ok=True)
(APP_DIR / "ori_bootstrap.py").write_text(BOOTSTRAP, encoding="utf-8")
(APP_DIR / "app.py").write_text(APP, encoding="utf-8")

get_ipython().run_line_magic("pip", "install -q streamlit")

for p in ("st_proc", "cf_proc"):          # tắt tiến trình cũ nếu chạy lại cell
    if p in globals():
        try: globals()[p].terminate()
        except Exception: pass

env = os.environ.copy()
env["ORI_PROJ"] = str(PROJ)
env["ORI_USE_RERANKER"] = "1" if USE_RERANKER else "0"
env["PYTHONUTF8"] = "1"

st_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", str(APP_DIR / "app.py"),
     "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
     "--browser.gatherUsageStats", "false"],
    cwd=APP_DIR, env=env, stdout=open(WORK / "streamlit.log", "w"), stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        if requests.get("http://localhost:8501/_stcore/health", timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    print((WORK / "streamlit.log").read_text()[-2000:])
    raise RuntimeError("Streamlit không khởi động được, xem log ở trên.")

cf_bin = WORK / "cloudflared"
if not cf_bin.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf_bin)
    cf_bin.chmod(cf_bin.stat().st_mode | stat.S_IEXEC)

cf_log = WORK / "cloudflared.log"
cf_proc = subprocess.Popen([str(cf_bin), "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                           stdout=open(cf_log, "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", cf_log.read_text())
    if m:
        url = m.group(0); break
    time.sleep(1)

print("🔗 Mở giao diện tại:", url or "(chưa lấy được link, xem " + str(cf_log) + ")")
print("Lần mở ĐẦU TIÊN sẽ nạp mô hình ~2,2 GB nên mất vài phút, cứ để trang chạy.")


## Phụ lục
- **Dọn dẹp:** chạy cell dưới để tắt Gateway.
- **MCP / giọng nói:** `mcp_server.py` (stdio) và `voice_main.py` (cần micro) không phù hợp để chạy trong notebook nên không đưa vào.
- **Dùng Redis thật:** bỏ cell đăng ký `rag.vector_store` ở Bước 3, điền `REDIS_URL`/`REDIS_INDEX_NAME` vào `.env`, cài `langchain-redis redis`, rồi chạy `python -m rag.ingest`.
- **Lưu mô hình đã huấn luyện:** tải thư mục `ner_model/phobert_ner_retrained` (Colab: *Files → work/.../ner_model*) trước khi runtime hết hạn.

In [ ]:
for name in ('gateway_proc', 'st_proc', 'cf_proc'):
    try:
        globals()[name].terminate(); print('Đã tắt', name)
    except Exception as e:
        print(name, e)
